# Task 2 - Step 4: Export results

Copies every table to `results/task2/` and every figure to `figures/task2/`, and builds one nested
`results/task2/results.json` (raw unrounded values, full per-epoch histories, configs, the freeze manifest and
the target-label access log).

In [1]:
# ---- Task 2 common header (identical in every Task 2 notebook) ----
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Locate the repository root (the folder containing shared/) and make it importable.
REPO = Path.cwd().resolve()
while not (REPO / "shared" / "pacs.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Run this notebook from inside the PA1 repository")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from shared import pacs, pacs_protocol as proto
from shared.config import load_config

T2 = REPO / "task2"
CFG_DIR = T2 / "configs"
SEED = 6304
# Smoke mode (env TASK2_SMOKE=1): 2 epochs x 5 updates per run, all outputs under _smoke/ folders,
# and notebook 03 uses RANDOM labels instead of loading the real Sketch labels.
SMOKE = os.environ.get("TASK2_SMOKE", "0") == "1"
SUB = "_smoke" if SMOKE else ""
RES = T2 / "results" / SUB
TAB, FIG = RES / "tables", RES / "figures"
DATA_TAB = T2 / "results" / "tables"      # data-preparation tables (same in smoke and real mode)
CKPT = T2 / "checkpoints" / SUB            # git-ignored
CACHE = T2 / "cache" / SUB                 # git-ignored
for p in [TAB, FIG, CKPT, CACHE]:
    p.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Config names (files in task2/configs). The main comparison uses lambda = 1 for DAN.
# DANN/CDAN with the manual's exact settings were unstable (DANN diverged, CDAN collapsed repeatedly). Two fixes
# were tried, both chosen from source-side evidence only (see task2/RUN_LOG.md):
#   1) discriminator lr x10 (2026-09-22, before target evaluation): CDAN stable, DANN diverged at epoch 10;
#   2) BatchNorm in the discriminator hidden layer, manual lr (post-evaluation rerun after root-cause diagnosis).
# Fix 2 passes the pre-set stability criteria for both and is the main DANN/CDAN row; all other runs are reported.
MAIN_RUNS = ["source_only", "dan", "dann_discbn", "cdan_discbn"]
ADAPT_MAIN = ["dan", "dann_discbn", "cdan_discbn"]
AS_SPECIFIED_ADV = ["dann", "cdan"]
DISCLR10_ADV = ["dann_disclr10", "cdan_disclr10"]
ADV_RUNS = ["dann_discbn", "cdan_discbn", "dann_disclr10", "cdan_disclr10", "dann", "cdan"]
STUDY_RUNS = ["dan_lambda0.1", "dan", "dan_lambda10"]          # DAN lambda in {0.1, 1, 10}
ALL_RUNS = MAIN_RUNS + DISCLR10_ADV + AS_SPECIFIED_ADV + ["dan_lambda0.1", "dan_lambda10"]
RUN_NAME = {c: load_config(CFG_DIR, c)["run_name"] for c in ALL_RUNS}
LABEL = {"source_only": "Source-only", "dan": "DAN (λ=1)", "dann_discbn": "DANN (disc BN)", "cdan_discbn": "CDAN (disc BN)",
         "dann_disclr10": "DANN (disc lr×10)",
         "cdan_disclr10": "CDAN (disc lr×10)", "dann": "DANN (as specified)", "cdan": "CDAN (as specified)",
         "dan_lambda0.1": "DAN (λ=0.1)", "dan_lambda10": "DAN (λ=10)"}

plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "legend.fontsize": 9, "savefig.dpi": 150})


def savefig(fig, stem):
    """Every figure is saved as PNG + PDF (never only displayed)."""
    fig.savefig(FIG / f"{stem}.png", bbox_inches="tight")
    fig.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    print("saved figure", stem)


print("REPO:", REPO, "| device:", DEVICE, "| smoke:", SMOKE)

REPO: C:\Users\afifh\Desktop\ATML\PA1 | device: cuda | smoke: False


In [2]:
import math, shutil
# Smoke outputs are never exported to the top-level results/ or figures/ folders.
R_OUT = (RES / "export") if SMOKE else (REPO / "results" / "task2")
F_OUT = (RES / "export" / "figures") if SMOKE else (REPO / "figures" / "task2")
R_OUT.mkdir(parents=True, exist_ok=True); F_OUT.mkdir(parents=True, exist_ok=True)
copied = []
for p in sorted(set(TAB.glob("task2_*")) | {DATA_TAB / "task2_data_summary.json", DATA_TAB / "task2_split_class_counts.csv"}):
    shutil.copy2(p, R_OUT / p.name); copied.append(p.name)
for p in sorted(FIG.glob("task2_*")):
    shutil.copy2(p, F_OUT / p.name); copied.append("figures/" + p.name)
shutil.copy2(proto.SPLIT_PATH, R_OUT / "task2_split_pacs_sketch_seed6304.json")
print(len(copied), "files copied")

30 files copied


In [3]:
def clean(o):
    if isinstance(o, float) and (math.isnan(o) or math.isinf(o)):
        return None
    if isinstance(o, dict):
        return {str(k): clean(v) for k, v in o.items()}
    if isinstance(o, list):
        return [clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        return o.item()
    return o

def csv(name):
    return pd.read_csv(TAB / name)

comp = csv("task2_method_comparison.csv").set_index("config")
res = {"task": "task2_unsupervised_domain_adaptation", "seed": SEED,
       "data": json.loads((DATA_TAB / "task2_data_summary.json").read_text()),
       "runs": {}, "controlled_study": csv("task2_controlled_study.csv").to_dict("records"),
       "freeze_manifest": json.loads((CACHE / "freeze_manifest.json").read_text()),
       "target_label_access_log": ([json.loads(l) for l in (REPO / "data" / "pacs" / "target_label_access_log.jsonl").read_text().splitlines()]
                            if (REPO / "data" / "pacs" / "target_label_access_log.jsonl").exists() else None),
       "_notes": {"separability": "held-out accuracy of balanced logistic regression (C=1), source-val vs Sketch, 70/30, seed 6304; 0.5 = chance",
                  "changes": "fractions (multiply by 100 for percentage points)"}}
cls_t, sep = csv("task2_class_analysis.csv"), csv("task2_domain_separability.csv").set_index("config")
for n in ALL_RUNS:
    d = CKPT / RUN_NAME[n]
    res["runs"][n] = {"label": LABEL[n], "config": load_config(CFG_DIR, n),
                      "training_summary": json.loads((d / "summary.json").read_text()),
                      "history": json.loads((d / "history.json").read_text()),
                      "comparison": comp.loc[n].to_dict(),
                      "domain_separability": sep.loc[n].to_dict(),
                      "per_class_target": cls_t[cls_t.config == n].set_index("class")[["target_acc", "change_vs_source_only", "n_target"]].to_dict("index")}
res["transfer_cases"] = csv("task2_transfer_cases.csv").to_dict("records")
res["prediction_flips"] = csv("task2_prediction_flips.csv").to_dict("records")
res["files"] = sorted(copied)
(R_OUT / "results.json").write_text(json.dumps(clean(res), indent=1))
print("wrote", R_OUT / "results.json")

wrote C:\Users\afifh\Desktop\ATML\PA1\results\task2\results.json


In [4]:
c = comp.loc[MAIN_RUNS]
print(c[["mean_acc", "mean_macro_f1", "target_acc", "target_macro_f1", "target_acc_change_vs_source_only", "domain_separability"]].round(4))
print(csv("task2_controlled_study.csv").round(4))

             mean_acc  mean_macro_f1  target_acc  target_macro_f1  \
config                                                              
source_only    0.9342         0.9317      0.6747           0.6718   
dan            0.9437         0.9428      0.6989           0.6373   
dann_discbn    0.9459         0.9443      0.6518           0.6642   
cdan_discbn    0.9387         0.9391      0.5185           0.4866   

             target_acc_change_vs_source_only  domain_separability  
config                                                              
source_only                            0.0000               0.9973  
dan                                    0.0242               0.8516  
dann_discbn                           -0.0229               1.0000  
cdan_discbn                           -0.1563               0.9973  
   lambda_mmd  mean_acc  mean_macro_f1  worst_acc  domain_separability  \
0         0.1    0.9393         0.9398     0.8951               0.9643   
1         1.0    0.9437